In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain.tools import tool

@tool
def write_file_tool(filename: str, content: str) -> str:
    """파일을 지정된 경로에 작성합니다. (이름: write_file)"""
    return f"파일 [{filename}]에 내용이 성공적으로 기록되었습니다."

@tool
def execute_sql_tool(query: str) -> str:
    """데이터베이스에서 SQL 쿼리를 실행합니다. (이름: execute_sql)"""
    return f"쿼리 [{query}]가 실행되었습니다. (영향을 받은 행: 1개)"

@tool
def read_data_tool(source: str) -> str:
    """지정된 소스에서 데이터를 읽어옵니다. (이름: read_data)"""
    return f"[{source}]로 부터 데이터를 성공적으로 불러왔습니다: [샘플 데이터]"

tools = [write_file_tool, execute_sql_tool, read_data_tool]

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file_tool": True, # "approve", "reject", "edit"
                "execute_sql_tool": {"allowed_decisions": ["approve", "reject"]}, # "approve", "reject"
                "read_data_tool": False # HITL(사용자 개입) 없이 즉시 실행
            },
            description_prefix="도구 실행: "
        )
    ],
    checkpointer=InMemorySaver(),
    system_prompt="모든 답변은 한국어로 작성해주세요."
)

In [18]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "abc 테이블의 모든 데이터를 삭제해줘"}]},
    config={"configurable": {"thread_id": "1"}}
)

In [19]:
response

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='c7b6d0de-7c69-416a-b4a8-07f07536b823'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc;"}'}, '__gemini_function_call_thought_signatures__': {'a68de522-2ea0-4f01-a134-bdc7f8f610a6': 'EjQKMgEMOdbHVC3PvWPO/ouRcNK3L7eg8pVGeMtPrO8VseRUzOfGUWGj/qtCUNyiAAv5FDUQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e10cd-5909-7331-967d-b2cca273b954-0', tool_calls=[{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc;'}, 'id': 'a68de522-2ea0-4f01-a134-bdc7f8f610a6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 21, 'total_tokens': 223, 'input_token_details': {'cache_read': 0}})],
 '__interrupt__': [Interrupt(value={'action_requests': [{'nam

In [20]:
response['__interrupt__']

[Interrupt(value={'action_requests': [{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc;'}, 'description': "도구 실행: \n\nTool: execute_sql_tool\nArgs: {'query': 'DELETE FROM abc;'}"}], 'review_configs': [{'action_name': 'execute_sql_tool', 'allowed_decisions': ['approve', 'reject']}]}, id='3543c6c0ba4874eeca777465401daafb')]

In [21]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        # resume={"decisions": [{"type": "approve"}]}
        resume={"decisions": [{"type": "reject"}]}
    ),
    config={"configurable": {"thread_id": "1"}}
)

In [22]:
response

{'messages': [HumanMessage(content='abc 테이블의 모든 데이터를 삭제해줘', additional_kwargs={}, response_metadata={}, id='c7b6d0de-7c69-416a-b4a8-07f07536b823'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'execute_sql_tool', 'arguments': '{"query": "DELETE FROM abc;"}'}, '__gemini_function_call_thought_signatures__': {'a68de522-2ea0-4f01-a134-bdc7f8f610a6': 'EjQKMgEMOdbHVC3PvWPO/ouRcNK3L7eg8pVGeMtPrO8VseRUzOfGUWGj/qtCUNyiAAv5FDUQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e10cd-5909-7331-967d-b2cca273b954-0', tool_calls=[{'name': 'execute_sql_tool', 'args': {'query': 'DELETE FROM abc;'}, 'id': 'a68de522-2ea0-4f01-a134-bdc7f8f610a6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 21, 'total_tokens': 223, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='User rejected the tool call for `execut